# ProBench Quick Start

This notebook demonstrates the main ProBench pipeline from prompt construction to inference and evaluation.

The workflow includes:

1. **Prompt preparation** for building the input dataset with prompts
2. **Inference** for generating model outputs
3. **Evaluation** for benchmarking and comparing generated solutions against baseline and reference runs

Use this notebook as a lightweight starting point for experimenting with different prompt settings, LLM providers, and evaluation configurations.

> **Note**
> For inference or evaluation runs, it is recommended to copy the printed commands and run them directly in a terminal.

## 1 · Prompt Config

In [2]:
DATASET_PATH  = "probench-swe/SWE-Pro"
OUTPUT_PATH   = "data/prompt_dataset/out.json"
TOKENIZER     = "cl100k_base"
MAX_TOKENS    = 200_000

## 2 · Build Oracle Prompts

In [4]:
import subprocess
import sys

cmd = [
    sys.executable,
    "-m",
    "probench.prep.prompt.prompt_oracle_builder",
    "--dataset_path", DATASET_PATH,
    "--output_path", OUTPUT_PATH,
    "--tokenizer", TOKENIZER,
    "--max_tokens", str(MAX_TOKENS)
]

print("Running:", " ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Running: /opt/anaconda3/bin/python -m probench.prep.prompt.prompt_oracle_builder --dataset_path probench-swe/SWE-Pro --output_path data/prompt_dataset/out.json --tokenizer cl100k_base --max_tokens 200000



## 3 · Build BM25 Prompts

In [23]:
import subprocess
import sys

#BM25_RETRIEVALS_PATH = ""

cmd = [
    sys.executable,
    "-m",
    "probench.prep.prompt.prompt_bm25_builder",
    "--dataset_path",         DATASET_PATH,
    "--bm25_retrievals_path", BM25_RETRIEVALS_PATH,
    "--output_path",          OUTPUT_PATH,
    "--tokenizer",            TOKENIZER,
    "--max_tokens",           str(MAX_TOKENS),
]

print("Running:", " ".join(cmd))
print("Using python:", sys.executable)

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Running: /Users/ezgisarikayak/Desktop/benchmark/probench_swe/bin/python -m probench.prep.prompt.prompt_bm25_builder --dataset_path data/dataset_base_21_02.json --bm25_retrievals_path retrieval_results_lucene_method_and_function_level/dataset_base_21_02/bm25_method_and_function_level_retrievals.jsonl --output_path data/prompt_dataset/out.json --tokenizer cl100k_base --max_tokens 200000 --granularity method_and_function_level
Using python: /Users/ezgisarikayak/Desktop/benchmark/probench_swe/bin/python



## 4 · Inspect

In [5]:
from probench.prep.data_loader import DatasetLoader

loader  = DatasetLoader(OUTPUT_PATH, mode="raw")
samples = list(loader)
print(f"Total samples: {len(samples)}")

Total samples: 102


In [6]:
# Preview one prompt
row = samples[0]
print("scenario_id:", row.get("scenario_id"))
print()
print(row["text"][:1000])

scenario_id: pandas-50778

You are an expert Python performance engineer. Your objective is to optimize pandas.Series.replace for speed and memory efficiency without altering its behavior, public APIs, or test expectations.

CONSTRAINTS:
- Preserve exact semantics and outputs.
- Preserve all public APIs and return types.
- Do not modify tests or test expectations.

OPTIMIZATION REQUEST:
Optimize Series.replace when using a large dict for to_replace
Optimization function: pandas.Series.replace


[start of pandas/core/internals/blocks.py]
### Full file context
```python
from __future__ import annotations

from functools import wraps
import re
from typing import (
    TYPE_CHECKING,
    Any,
    Callable,
    Iterable,
    Sequence,
    cast,
    final,
)

import numpy as np

from pandas._libs import (
    internals as libinternals,
    lib,
    writers,
)
from pandas._libs.internals import (
    BlockPlacement,
    BlockValuesRefs,
)
from pandas._libs.missing import NA
from pandas._libs.

In [7]:
from probench.prep.utils import prompt_preview

# Preview a specific scenario_id.
# If `scenario_id` is None or not found, the first sample is used.

id, text = prompt_preview(
    input_json=OUTPUT_PATH,
    selected_scenario_id=None  # or "some_scenario_id"
)

print("id:", id)
print("text:", text)

id: pandas-50778
text: You are an expert Python performance engineer. Your objective is to optimize pandas.Series.replace for speed and memory efficiency without altering its behavior, public APIs, or test expectations.

CONSTRAINTS:
- Preserve exact semantics and outputs.
- Preserve all public APIs and return types.
- Do not modify tests or test expectations.

OPTIMIZATION REQUEST:
Optimize Series.replace when using a large dict for to_replace
Optimization function: pandas.Series.replace


[start of pandas/core/internals/blocks.py]
### Full file context
```python
from __future__ import annotations

from functools import wraps
import re
from typing import (
    TYPE_CHECKING,
    Any,
    Callable,
    Iterable,
    Sequence,
    cast,
    final,
)

import numpy as np

from pandas._libs import (
    internals as libinternals,
    lib,
    writers,
)
from pandas._libs.internals import (
    BlockPlacement,
    BlockValuesRefs,
)
from pandas._libs.missing import NA
from pandas._libs.tsli

## 5 · Inference

Runs the inference pipeline on the prompt dataset built in the previous step.
Completions are saved to `inference/<run_id>/instances/<scenario_id>/completion.txt`.
Already-completed scenarios are skipped automatically on re-run.

> **Note on API keys**
>
> Before running inference, make sure the required API key is available as an environment variable or in a `.env` file in the project root.
>
> The current API key mapping is:
>
> - `openai_chat` -> `OPENAI_API_KEY`
> - `openai_responses` -> `OPENAI_API_KEY`
> - `hf` -> `HUGGINGFACE_API_KEY`
> - `ollama` -> `OLLAMA_API_KEY`
>
> Example `.env` file:
>
> ```bash
> OPENAI_API_KEY=your_openai_key_here
> HUGGINGFACE_API_KEY=your_hf_key_here
> OLLAMA_API_KEY=your_ollama_key_here
> ```

> **Adding a new LLM clients**
>
> The inference module uses a factory-based LLM client design under `inference/llm_client/`.
> Existing clients include:
>
> - `openai_chat_client.py`
> - `openai_responses_client.py`
> - `ollama_client.py`
> - `hf_client.py`
>
> All clients follow the same structure:
>
> 1. inherit from `BaseLLMClient`
> 2. implement `generate(prompt: str) -> str`
> 3. register the provider in `get_llm_client(...)`
> 4. add API key support in `_load_key_from_env(...)` if needed
>
> If you want to add another LLM client, follow the same recipe:
>
> - create a new client file under `inference/llm_client/`
> - implement a new class based on `BaseLLMClient`
> - add the provider branch in `get_llm_client(...)`
> - add a new environment variable entry in `_load_key_from_env(...)` if the provider needs authentication
>
> This keeps the inference pipeline modular and consistent across providers.

> **Note on running inference**
>
> The notebook prints the full command used to run inference.  
> For better performance and stability, we recommend copying this printed command and running it directly in a **terminal** instead of executing it inside the notebook.
>
> Running the command in a terminal is typically faster and avoids potential notebook-related overhead during long inference runs.

In [33]:
import subprocess
import sys

# Make sure to add your API key to a `.env` file or set it as an environment variable.

cmd = [
    sys.executable,
    "-m",
    "probench.inference.run_inference",
    "--dataset_path", OUTPUT_PATH,   # from prompt preparation cell
    "--provider", "openai_chat",
    "--model", "todo",
    "--out_root", "inference_test",
]

# Optional
# cmd += ["--max_instances", "5"]
# cmd += ["--tag", "attempt1"]
# cmd += ["--save_prompt"]
# cmd += ["--temperature", "0.2"]
# cmd += ["--top_p", "0.95"]
# cmd += ["--max_tokens", "4000"]

print("Running:", " ".join(cmd))
print("Using python:", sys.executable)

result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

Running: /Users/ezgisarikayak/Desktop/benchmark/probench_swe/bin/python -m probench.inference.run_inference --dataset_path data/prompt_dataset/out.json --provider openai_chat --model gpt-5-2025-08-07 --endpoint_url https://openai-openai_chat-msa-001333-swedencentral-01-freeexperiment.openai.azure.com/openai/v1 --out_root inference_test
Using python: /Users/ezgisarikayak/Desktop/benchmark/probench_swe/bin/python


KeyboardInterrupt: 

## 6 · Evaluation Harness

Runs the evaluation pipeline on the generated inference outputs.

For each scenario, the harness reconstructs the required setup, builds the corresponding Docker images, executes both the **baseline** and **LLM-generated** containers, and compares the results.

Depending on the configuration, the harness can also run the **gold/reference solution**.

The evaluation pipeline checks execution behavior, compares outcomes across versions, and collects scenario-level results in a reproducible containerized environment.

All outputs such as logs, reports, and evaluation artifacts are stored under: results/evals/<run_id>/

> **Note on running the evaluation harness**
>
> The notebook prints the full command used to run the evaluation harness.  
> For better performance and stability, we recommend copying this printed command and running it directly in a **terminal** instead of executing it inside the notebook.
>
> The evaluation pipeline builds Docker images, launches containers, and runs multiple scenario executions. Running these steps from a terminal is typically more stable and easier to monitor than executing them inside Jupyter.

> **Note on performance measurements**
>
> Even though the evaluation is executed inside Docker containers to improve reproducibility, performance measurements (e.g., runtime or resource usage) may still vary across machines. Differences in hardware, CPU architecture, background system load, operating system behavior, and container runtime environments can lead to small variations in measured results.
>
> Therefore, performance comparisons should primarily be interpreted **relative to the baseline within the same run and environment**, rather than as absolute measurements across different machines.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 
    "-m", 
    "probench.harness.run_evaluation",
    "--inference_run_dir", "inference/<run_id>",   # replace with your actual inference output directory, e.g. inference_test
    "--results_root", "results",
]

# Optional
# cmd += ["--scenario_id", "pandas-50778"]   # run a single scenario
# cmd += ["--with-gold-solution"]            # also run reference/oracle solution
# cmd += ["--force_rebuild"]                 # force Docker image rebuild
# cmd += ["--skip-baseline-if-done"]         # skip baseline run if results already exist
# cmd += ["--skip-reference-if-done"]        # skip reference run if results already exist

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

## 7 · Evaluation Harness: Aggregation and Reporting

This script aggregates one completed experiment run and produces:

- gate-funnel statistics
- per-metric improvement/regression/conflict summaries
- overall aggregate statistics
- `experiment_summary.json`

Performance numbers should be interpreted relative to the baseline within the same experiment environment, not as absolute machine-independent measurements.

In [ ]:
import subprocess
import sys
from pathlib import Path


EXPERIMENT_DIR = Path("outputs/experiments/latest")
OUTPUT_PATH = EXPERIMENT_DIR / "experiment_summary.json"

cmd = [
    sys.executable,
    "-m",
    "swe_pro.reporting.aggregate_experiments",
    "--experiment_dir", str(EXPERIMENT_DIR),
    "--output_path", str(OUTPUT_PATH),
    "--print_report",
]

print("Running:", " ".join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(
        f"Experiment aggregation failed with exit code {result.returncode}"
    )

print(f"\nSaved summary to: {OUTPUT_PATH}")